# EternaFold

In [ ]:
import os
import subprocess
import pandas as pd
import time
from pathlib import Path

In [ ]:
method_name = "EternaFold"
base = Path.cwd()

print(f"Base directory: {base}")

In [ ]:
tools_dir = (base.parent / 'tools').resolve()
repo_dir = tools_dir / 'EternaFold'
src_dir = repo_dir / 'src'
exe_path = src_dir / 'contrafold'
params_path = repo_dir / 'parameters' / 'EternaFoldParams.v1'

tools_dir.mkdir(parents=True, exist_ok=True)

if not repo_dir.exists():
    subprocess.run(
        ['git', 'clone', 'https://github.com/WaymentSteeleLab/EternaFold.git', str(repo_dir)],
        check=True,
    )

print(f'EternaFold repo: {repo_dir}')
print(f'EternaFold executable: {exe_path}')
print(f'EternaFold params: {params_path}')

In [ ]:
if not params_path.exists():
    raise FileNotFoundError(f'Missing parameter file: {params_path}')

makefile_path = src_dir / 'Makefile'
utilities_hpp = src_dir / 'Utilities.hpp'

makefile_text = makefile_path.read_text()
if '-fpermissive' not in makefile_text:
    makefile_text = makefile_text.replace('CXXFLAGS =', 'CXXFLAGS = -fpermissive', 1)
    makefile_path.write_text(makefile_text)

utilities_text = utilities_hpp.read_text()
if '#include <limits.h>' not in utilities_text:
    utilities_text = utilities_text.replace('#include <vector>\n', '#include <vector>\n#include <limits.h>\n', 1)
    utilities_hpp.write_text(utilities_text)

if not exe_path.exists():
    subprocess.run(['make', 'clean'], cwd=src_dir, check=True)
    subprocess.run(['make'], cwd=src_dir, check=True)

if not exe_path.exists():
    raise FileNotFoundError(f'Compilation finished but executable is missing: {exe_path}')

print('EternaFold is ready.')

In [ ]:
def read_virus_fasta(path: str):
    lines = [ln.strip() for ln in open(path, 'r').read().splitlines() if ln.strip() != '']
    records = []
    for i in range(0, len(lines), 3):
        header, seq, struct = lines[i], lines[i+1], lines[i+2]
        name = header[1:].strip()
        records.append((name, seq.strip(), struct.strip()))
    df = pd.DataFrame(records, columns=['name','sequence','structure']).set_index('name')
    return df

viruses = read_virus_fasta('../data/viruses.fasta')

selected_virus_keys = None

if selected_virus_keys is None:
    virus_ids = list(viruses.index)
else:
    tmp = []
    for k in selected_virus_keys:
        if isinstance(k, int):
            tmp.append(viruses.index[k])
        else:
            tmp.append(str(k))
    virus_ids = tmp

In [ ]:
def run_folding(fasta_name):
    out_file_name = "EternaFold_folded_seq.fasta"
    tmp_stdout = Path("EternaFold_tmp")

    if tmp_stdout.exists():
        tmp_stdout.unlink()

    with open(tmp_stdout, "w") as tmp_handle:
        result = subprocess.run(
            [str(exe_path), 'predict', fasta_name, '--params', str(params_path)],
            stdout=tmp_handle,
            stderr=subprocess.DEVNULL,
            check=False,
        )

    if result.returncode != 0 or not tmp_stdout.exists():
        return None

    with open(out_file_name, "w") as fout:
        if tmp_stdout.exists():
            for line in tmp_stdout.open():
                if ">structure" not in line and line.strip() != "Predicting using MEA estimator.":
                    fout.write(line)
            tmp_stdout.unlink()
           
    return out_file_name

In [ ]:
os.makedirs('../prediction', exist_ok=True)
out_fasta_name = '../prediction/EternaFold.fasta'
if os.path.exists(out_fasta_name):
    os.remove(out_fasta_name)

print(f"{' ':3}\t{'virus':<20}\t{'len':<5}\t{'time'}")
for i, vid in enumerate(virus_ids):
    start_time = time.time()
    seq = viruses.loc[vid]['sequence']
    print(f"{i+1:3d}/{len(virus_ids)}\t{vid:<20}\t{len(seq):<5}\t", end='', flush=True)

    with open("EternaFold_tmp.fasta", "w") as ofile:
        ofile.write(f">{vid}\n{seq}\n")

    dot_file_name = run_folding("EternaFold_tmp.fasta")

    if dot_file_name and os.path.exists(dot_file_name):
        with open(dot_file_name, 'r') as src, open(out_fasta_name, 'a') as dst:
            dst.write(src.read())

    if os.path.exists("EternaFold_tmp.fasta"):
        os.remove("EternaFold_tmp.fasta")
    if dot_file_name and os.path.exists(dot_file_name):
        os.remove(dot_file_name)

    elapsed_time = time.time() - start_time
    print(f"{elapsed_time: .1f} s")